In [1]:
import sys
import os
from pathlib import Path

# Ensure we're in the correct working directory
os.chdir(Path(__file__).parent if '__file__' in dir() else Path.cwd())
print(f"Working directory: {os.getcwd()}")
print(f"Python: {sys.executable}")

# Install required packages if missing
import subprocess
for pkg in ['nbformat', 'nbclient']:
    try:
        __import__(pkg)
        print(f"✓ {pkg} already installed")
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
        print(f"✓ {pkg} installed")

print("\nReady to proceed!")

Working directory: /Users/hari.durai/automationProject/ai-agent-langgraph
Python: /Users/hari.durai/automationProject/ai-agent-langgraph/.venv2/bin/python
Installing nbformat...



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


✓ nbformat installed
Installing nbclient...
✓ nbclient installed

Ready to proceed!



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# Execute the existing notebook and capture stdout outputs (robust)
from nbformat import read
from nbclient import NotebookClient
from pathlib import Path
import re
import traceback
nb_path = Path('testCaseGeneratorAgent.ipynb')
if not nb_path.exists():
    raise FileNotFoundError(f"Notebook not found: {nb_path}")

# Execute with protection to capture errors
testcases_text = None
try:
    nb = read(str(nb_path), as_version=4)
    client = NotebookClient(nb, timeout=600, kernel_name='python3')
    print('Executing notebook (this may take a while)...')
    client.execute()
    
    # Collect all stream outputs into one string
    all_stream = ''
    for c in nb.cells:
        outputs = c.get('outputs', [])
        for o in outputs:
            if o.get('output_type') == 'stream':
                all_stream += o.get('text', '')
    
    # Try to extract the "Generated Test Cases:" section
    m = re.search(r'Generated Test Cases:\n(.*?)(?:\n={10,}|$)', all_stream, flags=re.DOTALL)
    if not m:
        print('Could not find "Generated Test Cases:" in notebook output.')
        print('--- Notebook stream output (truncated) ---')
        print(all_stream[:4000])
        # Attempt fallback to embedded test_cases in file
        m2 = re.search(r'test_cases\s*=\s*"""(.*?)"""', open(nb_path,'r',encoding='utf-8').read(), flags=re.DOTALL)
        if m2:
            testcases_text = m2.group(1).strip()
            print('Using embedded test_cases from notebook file (fallback)')
        else:
            raise RuntimeError('Test cases not found in executed notebook output')
    else:
        testcases_text = m.group(1).strip()
        print(f'Extracted {len(testcases_text)} characters of test cases')
        
except Exception as e:
    print('Error executing notebook:')
    traceback.print_exc()
    # If we haven't already extracted testcases_text, try to salvage from file
    if testcases_text is None:
        try:
            txt = open(nb_path, 'r', encoding='utf-8').read()
            m2 = re.search(r'test_cases\s*=\s*"""(.*?)"""', txt, flags=re.DOTALL)
            if m2:
                testcases_text = m2.group(1).strip()
                print('Extracted embedded test_cases from notebook file (fallback)')
            else:
                raise
        except Exception:
            print('Could not recover test cases from notebook file; re-raise original error')
            raise

# Ensure testcases_text is defined before proceeding
if testcases_text is None:
    raise RuntimeError('Failed to extract test cases from notebook')
print(f'Ready to generate Java tests from {len(testcases_text)} characters of test cases')

/Users/hari.durai/automationProject/ai-agent-langgraph/.venv2/lib/python3.14/site-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


Executing notebook (this may take a while)...
Error executing notebook:
Could not recover test cases from notebook file; re-raise original error


Traceback (most recent call last):
  File "/var/folders/q4/q6252jh957x61tq2gbsdjb7w0000gq/T/ipykernel_41058/443730530.py", line 17, in <module>
    client.execute()
    ~~~~~~~~~~~~~~^^
  File "/Users/hari.durai/automationProject/ai-agent-langgraph/.venv2/lib/python3.14/site-packages/jupyter_core/utils/__init__.py", line 171, in wrapped
    return _runner_map[name].run(inner)
           ~~~~~~~~~~~~~~~~~~~~~^^^^^^^
  File "/Users/hari.durai/automationProject/ai-agent-langgraph/.venv2/lib/python3.14/site-packages/jupyter_core/utils/__init__.py", line 128, in run
    return fut.result(None)
           ~~~~~~~~~~^^^^^^
  File "/opt/homebrew/Cellar/python@3.14/3.14.2_1/Frameworks/Python.framework/Versions/3.14/lib/python3.14/concurrent/futures/_base.py", line 450, in result
    return self.__get_result()
           ~~~~~~~~~~~~~~~~~^^
  File "/opt/homebrew/Cellar/python@3.14/3.14.2_1/Frameworks/Python.framework/Versions/3.14/lib/python3.14/concurrent/futures/_base.py", line 395, in __get_r

CellExecutionError: An error occurred while executing the following cell:
------------------
ISSUE_KEY = "MBA-8"

info = get_issue_summary_and_description(ISSUE_KEY)

print("=" * 60)
print(f"Generating Test Cases for: {ISSUE_KEY}")
print("=" * 60)
print(f"\nSummary: {info['summary']}")
print(f"\nDescription preview (first 300 chars):\n{info['description'][:300]}...")

try:
    result = test_cases_chain.invoke({
        "key": ISSUE_KEY,
        "summary": info["summary"],
        "description": info["description"]
    })
    test_cases = result.content.strip()
except Exception as e:
    print(f"\nNote: Could not connect to OpenAI API: {type(e).__name__}")
    print("Using demo test cases instead:\n")
    test_cases = """TC001 - Positive: Create Policy with Valid Details (No Previous Claims)
Preconditions: User is on the motor insurance policy creation page
Steps:
1. Enter Customer Name: John Smith
2. Enter DOB: 15/01/1985
3. Enter Contact: +44-7900-123456
4. Enter Car Registration: AB2

In [ ]:
# Save extracted test cases to a file and run the generator script
from pathlib import Path
import subprocess, sys
out_file = Path('generated_testcases.txt')
out_file.write_text(testcases_text, encoding='utf-8')
print(f'Wrote test cases to: {out_file.resolve()}')
# Default target - update if you want a different Playwright Java project
default_target = '/Users/hari.durai/automationProject/agentic-ai-automation-test'
print('Running generator script to create Java tests (this invokes the local script)...')
cmd = [sys.executable, 'scripts/generate_playwright_java_tests.py', '--testcases-file', str(out_file), '--issue-key', 'MBA-8', '--target', default_target]
try:
    subprocess.check_call(cmd)
    print('Generator script completed successfully')
except subprocess.CalledProcessError as e:
    print('Generator script failed with exit code', e.returncode)
    print('Command:', ' '.join(cmd))
    raise

**Notes & next steps**

- Open the generated Java file(s) under `<target>/src/test/java/ai/generated/` and refine selectors/assertions.
- If the target project is outside this machine or you do not want the notebook to write there, change `default_target` before running the generation cell.
- If execution fails due to environment differences, run the notebook kernel from the same virtualenv that can access project paths and credentials.